# Gold - Tablas de consumo para Power BI
Este notebook construye tablas y vista de resumen para analitica comparativa en Power BI.

Prerequisito: silver.POKEMON_ENRIQUECIDO debe existir.
Parametro esperado en Databricks SQL: catalogo.

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "silver")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "golden")


In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")



In [0]:
%sql
SELECT
  zona,
  numero_generacion,
  COUNT(*) AS total_pokemons,
  SUM(CASE WHEN legendario = 1 THEN 1 ELSE 0 END) AS total_legendarios,
  ROUND(AVG(total_base), 2) AS promedio_total_base,
  ROUND(AVG(ataque), 2) AS promedio_ataque,
  ROUND(AVG(defensa), 2) AS promedio_defensa,
  ROUND(AVG(velocidad), 2) AS promedio_velocidad,
  ROUND(AVG(tasa_captura), 2) AS promedio_tasa_captura
FROM ${catalogo}.${container}.POKEMON_ENRIQUECIDO GROUP BY zona, numero_generacion


## Tabla Gold: PBI_RESUMEN_ZONA
**Objetivo:** generar un resumen ejecutivo por zona y generacion con metricas clave para dashboards.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.${esquema}.PBI_RESUMEN_ZONA
AS
SELECT
  zona,
  numero_generacion,
  COUNT(*) AS total_pokemons,
  SUM(CASE WHEN legendario = 1 THEN 1 ELSE 0 END) AS total_legendarios,
  ROUND(AVG(total_base), 2) AS promedio_total_base,
  ROUND(AVG(ataque), 2) AS promedio_ataque,
  ROUND(AVG(defensa), 2) AS promedio_defensa,
  ROUND(AVG(velocidad), 2) AS promedio_velocidad,
  ROUND(AVG(tasa_captura), 2) AS promedio_tasa_captura
FROM ${catalogo}.${container}.POKEMON_ENRIQUECIDO GROUP BY zona, numero_generacion;

## Tabla Gold: PBI_TOP_HABILIDADES_ZONA
**Objetivo:** rankear habilidades mas representativas por zona y generacion.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.${esquema}.PBI_TOP_HABILIDADES_ZONA
AS
WITH abilities_by_zone AS (
  SELECT
    pe.zona,
    pe.numero_generacion,
    a.nombre_habilidad AS habilidad,
    COUNT(DISTINCT pe.pokemon_id) AS pokemons_con_habilidad
  FROM ${catalogo}.silver.POKEMON_ENRIQUECIDO pe
  INNER JOIN ${catalogo}.bronze.POKEMON_ABILITIES pa
    ON pe.pokemon_id = pa.pokemon_id
  INNER JOIN ${catalogo}.bronze.ABILITIES a
    ON pa.habilidad_id = a.habilidad_id
  GROUP BY pe.zona, pe.numero_generacion, a.nombre_habilidad
), ranked AS (
  SELECT
    zona,
    numero_generacion,
    habilidad,
    pokemons_con_habilidad,
    DENSE_RANK() OVER (
      PARTITION BY zona, numero_generacion
      ORDER BY pokemons_con_habilidad DESC, habilidad ASC
    ) AS ranking_habilidad
  FROM abilities_by_zone
), totals AS (
  SELECT
    zona,
    numero_generacion,
    COUNT(DISTINCT pokemon_id) AS total_pokemons_zona_generacion
  FROM ${catalogo}.silver.POKEMON_ENRIQUECIDO
  GROUP BY zona, numero_generacion
)
SELECT
  r.zona,
  r.numero_generacion,
  r.habilidad,
  r.pokemons_con_habilidad,
  t.total_pokemons_zona_generacion,
  ROUND((r.pokemons_con_habilidad * 100.0) / t.total_pokemons_zona_generacion, 2) AS porcentaje_presencia,
  r.ranking_habilidad
FROM ranked r
INNER JOIN totals t
  ON r.zona = t.zona
 AND r.numero_generacion = t.numero_generacion
WHERE r.ranking_habilidad <= 10;

## Tabla Gold: PBI_COMPARATIVO_TIPOS_ZONA
**Objetivo:** comparar composicion y desempeno por tipo entre zonas y generaciones.

In [0]:
%sql
    
CREATE OR REPLACE TABLE ${catalogo}.${esquema}.PBI_COMPARATIVO_TIPOS_ZONA
AS
WITH pokemon_types AS (
  SELECT
    src.pokemon_id,
    src.zona,
    src.numero_generacion,
    src.legendario,
    src.total_base,
    exploded_tipo_id AS tipo_id
  FROM (
    SELECT
      pe.*,
      explode(array(pe.tipo1_id, pe.tipo2_id)) AS exploded_tipo_id
    FROM ${catalogo}.silver.POKEMON_ENRIQUECIDO pe
  ) src
  WHERE exploded_tipo_id IS NOT NULL
)
SELECT
  pt.zona,
  pt.numero_generacion,
  t.nombre_tipo AS tipo,
  COUNT(DISTINCT pt.pokemon_id) AS total_pokemons_tipo,
  ROUND(AVG(pt.total_base), 2) AS promedio_total_base_tipo,
  ROUND(100.0 * AVG(CASE WHEN pt.legendario = 1 THEN 1.0 ELSE 0.0 END), 2) AS porcentaje_legendarios
FROM pokemon_types pt
INNER JOIN ${catalogo}.bronze.TYPES t
  ON pt.tipo_id = t.tipo_id
GROUP BY pt.zona, pt.numero_generacion, t.nombre_tipo;

## Tabla Gold: PBI_EFECTIVIDAD_DEFENSIVA_ZONA
**Objetivo:** medir vulnerabilidad o resistencia defensiva promedio por tipo atacante.

In [0]:
%sql
CREATE OR REPLACE TABLE ${catalogo}.${esquema}.PBI_EFECTIVIDAD_DEFENSIVA_ZONA
AS
SELECT
  pe.zona,
  pe.numero_generacion,
  ta.nombre_tipo AS tipo_atacante,
  ROUND(AVG(pte.multiplicador), 4) AS multiplicador_promedio,
  CASE
    WHEN AVG(pte.multiplicador) >= 1.5 THEN 'Alta vulnerabilidad'
    WHEN AVG(pte.multiplicador) <= 0.75 THEN 'Alta resistencia'
    ELSE 'Balanceado'
  END AS clasificacion_defensiva
FROM ${catalogo}.silver.POKEMON_ENRIQUECIDO pe
INNER JOIN ${catalogo}.bronze.POKEMON_TYPE_EFFECTIVENESS pte
  ON pe.pokemon_id = pte.pokemon_id
INNER JOIN ${catalogo}.bronze.TYPES ta
  ON pte.tipo_atacante_id = ta.tipo_id
GROUP BY pe.zona, pe.numero_generacion, ta.nombre_tipo;

## Vista Gold: PBI_POKEMON_DETALLE
**Objetivo:** exponer detalle para drill-through y filtros avanzados en Power BI.

In [0]:
%sql
CREATE OR REPLACE VIEW ${catalogo}.${esquema}.PBI_POKEMON_DETALLE
AS
SELECT
  pokemon_id,
  numero_pokedex,
  pokemon_nombre,
  zona,
  numero_generacion,
  tipo_primario,
  tipo_secundario,
  nombre_tipo_experiencia,
  categoria_legendario,
  categoria_mega,
  total_base,
  ataque,
  defensa,
  ataque_especial,
  defensa_especial,
  velocidad,
  tasa_captura
FROM ${catalogo}.silver.POKEMON_ENRIQUECIDO;

## Diccionario de objetos Gold para Power BI
### gold.PBI_RESUMEN_ZONA
- **Grano:** zona + numero_generacion.
- **Uso recomendado:** tarjetas KPI y tendencias por region.

### gold.PBI_TOP_HABILIDADES_ZONA
- **Grano:** zona + numero_generacion + habilidad.
- **Uso recomendado:** barras Top N y matrices de afinidad.

### gold.PBI_COMPARATIVO_TIPOS_ZONA
- **Grano:** zona + numero_generacion + tipo.
- **Uso recomendado:** comparativos de tipos por geografia/generacion.

### gold.PBI_EFECTIVIDAD_DEFENSIVA_ZONA
- **Grano:** zona + numero_generacion + tipo_atacante.
- **Uso recomendado:** mapas de calor de vulnerabilidad defensiva.

### gold.PBI_POKEMON_DETALLE (vista)
- **Grano:** 1 fila por pokemon_id.
- **Uso recomendado:** detalle y drill-through desde visuales agregados.

In [0]:
%sql
SELECT 'PBI_RESUMEN_ZONA' AS tabla, COUNT(*) AS filas FROM ${catalogo}.${esquema}.PBI_RESUMEN_ZONA
UNION ALL
SELECT 'PBI_TOP_HABILIDADES_ZONA' AS tabla, COUNT(*) AS filas FROM ${catalogo}.${esquema}.PBI_TOP_HABILIDADES_ZONA
UNION ALL
SELECT 'PBI_COMPARATIVO_TIPOS_ZONA' AS tabla, COUNT(*) AS filas FROM ${catalogo}.${esquema}.PBI_COMPARATIVO_TIPOS_ZONA
UNION ALL
SELECT 'PBI_EFECTIVIDAD_DEFENSIVA_ZONA' AS tabla, COUNT(*) AS filas FROM ${catalogo}.${esquema}.PBI_EFECTIVIDAD_DEFENSIVA_ZONA;